# 🦺 Construction Site PPE Violation Detector
## Exploratory Data Analysis & Baseline Evaluation

This notebook walks through:
1. Dataset statistics and class distribution
2. Sample image visualisation with ground-truth annotations
3. Model baseline evaluation
4. Results plots (mAP, Precision-Recall curve)

In [ ]:
import os
import sys
import cv2
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as patches
from pathlib import Path
from collections import Counter

# Add src/ to path
sys.path.insert(0, '../src')

plt.style.use('seaborn-v0_8-whitegrid')
print('Libraries loaded ✓')

## 1. Dataset Statistics

In [ ]:
# ── Count images and annotations per split ───────────────────────────────────
DATA_ROOT = Path('../data/raw')
SPLITS = ['train', 'valid', 'test']

CLASS_NAMES = [
    'Hardhat', 'Mask', 'NO-Hardhat', 'NO-Mask',
    'NO-Safety Vest', 'Person', 'Safety Cone',
    'Safety Vest', 'machinery', 'vehicle'
]

stats = {}
class_counts = Counter()

for split in SPLITS:
    img_dir = DATA_ROOT / split / 'images'
    lbl_dir = DATA_ROOT / split / 'labels'
    
    if not img_dir.exists():
        print(f'[SKIP] {split} – directory not found (run download_data.py first)')
        continue
    
    imgs  = list(img_dir.glob('*.jpg')) + list(img_dir.glob('*.png'))
    lbls  = list(lbl_dir.glob('*.txt')) if lbl_dir.exists() else []
    stats[split] = {'images': len(imgs), 'labels': len(lbls)}
    
    # Count per-class instances
    for lbl_file in lbls:
        with open(lbl_file) as f:
            for line in f:
                cls_id = int(line.split()[0])
                class_counts[CLASS_NAMES[cls_id]] += 1

print('Dataset split summary:')
for split, s in stats.items():
    print(f"  {split:6s}: {s['images']:4d} images | {s['labels']:4d} label files")

In [ ]:
# ── Plot class distribution ──────────────────────────────────────────────────
if class_counts:
    fig, ax = plt.subplots(figsize=(12, 5))
    classes = list(class_counts.keys())
    counts  = list(class_counts.values())
    colors  = ['#e74c3c' if 'NO' in c else '#2ecc71' for c in classes]
    
    bars = ax.bar(classes, counts, color=colors, edgecolor='white', linewidth=0.8)
    ax.set_title('Class Distribution in Dataset', fontsize=14, fontweight='bold')
    ax.set_xlabel('Class')
    ax.set_ylabel('Number of Instances')
    plt.xticks(rotation=30, ha='right')
    
    for bar, count in zip(bars, counts):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 5,
                str(count), ha='center', va='bottom', fontsize=9)
    
    ax.legend(handles=[
        patches.Patch(color='#e74c3c', label='Violation class'),
        patches.Patch(color='#2ecc71', label='Safe / Other class'),
    ])
    plt.tight_layout()
    plt.savefig('../outputs/reports/class_distribution.png', dpi=150)
    plt.show()
    print('Plot saved → outputs/reports/class_distribution.png')
else:
    print('[INFO] No labels found – skipping class distribution plot.')
    print('       Download the dataset first: python src/download_data.py')

## 2. Visualise Sample Annotations

In [ ]:
def show_annotated_sample(img_path: Path, lbl_path: Path,
                           class_names: list, ax=None):
    """Draw YOLO bounding boxes on an image."""
    img_bgr = cv2.imread(str(img_path))
    img_rgb = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)
    h, w    = img_bgr.shape[:2]
    
    if ax is None:
        _, ax = plt.subplots(1, 1, figsize=(8, 6))
    ax.imshow(img_rgb)
    
    if lbl_path.exists():
        with open(lbl_path) as f:
            for line in f:
                parts = line.strip().split()
                if len(parts) < 5:
                    continue
                cls_id, cx, cy, bw, bh = int(parts[0]), *map(float, parts[1:5])
                x1 = int((cx - bw/2) * w)
                y1 = int((cy - bh/2) * h)
                bw_px = int(bw * w)
                bh_px = int(bh * h)
                color = '#e74c3c' if 'NO' in class_names[cls_id] else '#2ecc71'
                rect  = patches.Rectangle((x1, y1), bw_px, bh_px,
                                           linewidth=2, edgecolor=color, facecolor='none')
                ax.add_patch(rect)
                ax.text(x1, y1-4, class_names[cls_id],
                        color='white', fontsize=7, fontweight='bold',
                        bbox=dict(facecolor=color, alpha=0.8, pad=1, edgecolor='none'))
    ax.axis('off')
    ax.set_title(img_path.name, fontsize=9)


# Show up to 6 samples from training set
train_img_dir = DATA_ROOT / 'train' / 'images'
train_lbl_dir = DATA_ROOT / 'train' / 'labels'

if train_img_dir.exists():
    sample_imgs = sorted(train_img_dir.glob('*.jpg'))[:6]
    fig, axes = plt.subplots(2, 3, figsize=(15, 10))
    for img_path, ax in zip(sample_imgs, axes.flatten()):
        lbl_path = train_lbl_dir / (img_path.stem + '.txt')
        show_annotated_sample(img_path, lbl_path, CLASS_NAMES, ax=ax)
    plt.suptitle('Sample Training Images with Ground-Truth Annotations',
                 fontsize=14, fontweight='bold')
    plt.tight_layout()
    plt.savefig('../outputs/reports/sample_annotations.png', dpi=150)
    plt.show()
else:
    print('[INFO] Training images not found. Showing placeholder message.')
    print('       Run: python src/download_data.py --dest data/raw')

## 3. Baseline Model Evaluation

In [ ]:
# Run the evaluate.py demo (works without a real dataset)
from evaluate import evaluate_detections, compute_iou

# Synthetic ground-truth and predictions
ground_truths = [
    {'image_id': 'img1', 'label': 'No-Helmet',    'bbox': [100,100,200,200]},
    {'image_id': 'img1', 'label': 'Person',        'bbox': [50, 50, 300,400]},
    {'image_id': 'img2', 'label': 'No-Vest',       'bbox': [120,120,220,300]},
    {'image_id': 'img3', 'label': 'Helmet',        'bbox': [80, 80,160,160]},
    {'image_id': 'img3', 'label': 'Safety-Vest',   'bbox': [70, 160,200,400]},
]

predictions = [
    {'image_id': 'img1', 'label': 'No-Helmet', 'confidence': 0.91, 'bbox': [105,105,205,205]},
    {'image_id': 'img1', 'label': 'Person',    'confidence': 0.85, 'bbox': [ 55, 55,305,405]},
    {'image_id': 'img2', 'label': 'No-Vest',   'confidence': 0.78, 'bbox': [115,115,215,295]},
    {'image_id': 'img2', 'label': 'No-Helmet', 'confidence': 0.55, 'bbox': [  0,  0, 20, 20]},  # FP
    {'image_id': 'img3', 'label': 'Helmet',    'confidence': 0.92, 'bbox': [ 83, 83,163,163]},
    {'image_id': 'img3', 'label': 'Safety-Vest','confidence': 0.80,'bbox': [ 72,162,202,402]},
]

results = evaluate_detections(predictions, ground_truths)

print('=' * 45)
print('  Evaluation Results (Synthetic Demo)')
print('=' * 45)
for cls, ap in results['per_class_ap'].items():
    print(f'  AP [{cls:>14s}] : {ap:.4f}')
print(f"\n  mAP@0.5   : {results['mAP']:.4f}")
print(f"  Precision : {results['precision']:.4f}")
print(f"  Recall    : {results['recall']:.4f}")
print(f"  F1-Score  : {results['f1']:.4f}")
print('=' * 45)

In [ ]:
# ── Bar chart of per-class AP ────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(8, 4))
classes = list(results['per_class_ap'].keys())
aps     = list(results['per_class_ap'].values())
colors  = ['#e74c3c' if 'No' in c else '#3498db' for c in classes]

bars = ax.bar(classes, aps, color=colors, edgecolor='white')
ax.axhline(results['mAP'], color='orange', linestyle='--',
            linewidth=1.5, label=f"mAP = {results['mAP']:.3f}")
ax.set_ylim(0, 1.05)
ax.set_title('Per-Class Average Precision (mAP@0.5)', fontweight='bold')
ax.set_ylabel('AP')
ax.legend()

for bar, ap in zip(bars, aps):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01,
            f'{ap:.2f}', ha='center', fontsize=9)

plt.tight_layout()
os.makedirs('../outputs/reports', exist_ok=True)
plt.savefig('../outputs/reports/per_class_ap.png', dpi=150)
plt.show()
print('Plot saved → outputs/reports/per_class_ap.png')

## 4. IoU Demo

In [ ]:
# ── Visualise IoU between prediction and ground truth ────────────────────────
gt_box   = [100, 100, 200, 200]
pred_box = [120, 110, 220, 210]
iou_val  = compute_iou(pred_box, gt_box)

fig, ax = plt.subplots(figsize=(5, 5))
ax.set_xlim(60, 260); ax.set_ylim(60, 260); ax.invert_yaxis()

gt_rect   = patches.Rectangle((gt_box[0], gt_box[1]),
                                gt_box[2]-gt_box[0], gt_box[3]-gt_box[1],
                                linewidth=2, edgecolor='green',
                                facecolor='green', alpha=0.3, label='Ground Truth')
pred_rect = patches.Rectangle((pred_box[0], pred_box[1]),
                                pred_box[2]-pred_box[0], pred_box[3]-pred_box[1],
                                linewidth=2, edgecolor='red',
                                facecolor='red', alpha=0.3, label='Prediction')
ax.add_patch(gt_rect)
ax.add_patch(pred_rect)
ax.legend(loc='upper left')
ax.set_title(f'IoU = {iou_val:.4f}', fontsize=13, fontweight='bold')
ax.set_xlabel('x (pixels)'); ax.set_ylabel('y (pixels)')
plt.tight_layout()
plt.savefig('../outputs/reports/iou_demo.png', dpi=150)
plt.show()

---
**End of EDA Notebook** · All plots saved to `outputs/reports/`